# Hardware-Aware Model Architecture & Execution Mapper

End-to-end demo: profile **Qwen3-8B** on an NVIDIA H100 GPU and visualize across all 3 tabs.

- **Tab 1** — Workload Dashboard (per-request TTFT, decode, queue, GPU kernels)
- **Tab 2** — Request Inspector (nn.Module → op → kernel hierarchy with per-request variance)
- **Tab 3** — Architecture (one-time model structure with layer collapse)

All commands run on the **GPU host** unless noted otherwise.

## Prerequisites
# 🚀 vLLM Architecture & Profiling Map — Setup & Exploration
you can skip this step if you already have vllm installed and configure or if you want to 
follow this guide: https://docs.vllm.ai/en/latest/getting_started/quickstart/#nvidia-cuda
# 🚀 vLLM Architecture & Profiling Map — Setup & Exploration
### 1. Clone vllm repo
```bash
if [ ! -d "vllm" ]; then
  git clone https://github.com/vllm-project/vllm.git
fi
```

### 2. Create virtual environment
```bash
uv venv --python 3.12 --seed
source .venv/bin/activate
```

### 3. Install packages
```bash
uv pip install torch --torch-backend=auto 
uv pip install setuptools wheel setuptools-rust
uv pip install setuptools_scm ninja cmake numpy
cd vllm
MAX_JOBS=12 pip install -vv --no-build-isolation -e . # install it from source 
cd ../
```

### 4. System tools
- **nsys** (NVIDIA Nsight Systems) — comes with the CUDA toolkit (https://developer.nvidia.com/cuda-12-9-0-download-archive)
- Verify: `nsys --version` and `nvidia-smi`


---
## Tab 2: Per-Request Profiling (Request Inspector)

Single terminal, offline mode. Produces `per_request_profiles.json`.

`profile_serve.py` runs the model with `vllm.LLM` (offline, no server needed) and profiles each prompt individually using `torch.profiler`. Each prompt produces a Chrome trace (`.json.gz`) containing the full `nn.Module` → CPU op → GPU kernel hierarchy with real timing.

`profiler_to_diagram.py --per-request-dir` then reads these raw traces and builds a single merged module tree with per-request timing arrays. It correlates CPU ops to GPU kernels (via `External_id` → `cuda_runtime.correlation`, with timestamp-based fallback), collapses repeated decoder layers into a `×N` summary while preserving individual layer data for drill-down, and computes cross-layer variance stats.

**Why compress to JSON?** Raw Chrome traces are 50-100 MB each (171K+ events for Qwen3-8B). The processed `per_request_profiles.json` is ~1 MB — only the module tree structure, timing, and kernel mappings that the viewer needs. This makes SCP fast and the browser responsive.

In [ ]:
#Set model
import os
os.chdir("../../")
os.environ["MODEL"] = "Qwen/Qwen3-8B"

# Profile: accepts same dataset flags as vllm bench serve
#   --num-prompts: how many prompts to profile (default: 3)
#   --input-len / --output-len: token lengths
#   --dataset-name: random (default), sharegpt, sonnet, hf, custom
!python vllm_arch_map/profile_serve.py $MODEL \
    --num-prompts 3 --input-len 1024

(MainProcess pid=640447) INFO 10-17 01:47:43 [api_utils.py:286] non-default args: {'tokenizer': 'Qwen/Qwen3-8B', 'gpu_memory_utilization': 0.5, 'disable_log_stats': True, 'enforce_eager': True, 'profiler_config': ProfilerConfig(profiler='torch', torch_profiler_dir='/root/lkesem/vllm_arch_map/traces/per_request/profiles-2fb4eb59-f6ff-4fb6-aef9-1475a71bad2b', torch_profiler_activities=None, proton_profiler_dir='', proton_context='shadow', proton_data='tree', proton_backend=None, proton_mode=None, proton_hook=None, proton_output_format=None, proton_graph_attribution=False, torch_profiler_with_stack=True, torch_profiler_with_flops=False, torch_profiler_use_gzip=True, torch_profiler_dump_cuda_time_total=True, torch_profiler_record_shapes=True, torch_profiler_with_memory=False, capture_torch_profiler=False, detailed_trace_annotation=False, ignore_frontend=False, delay_iterations=0, max_iterations=0, warmup_iterations=0, active_iterations=5, wait_iterations=0), 'model': 'Qwen/Qwen3-8B'}
(Main

In [ ]:
# Verify traces were generated
!ls -lh traces/per_request/request_*.json*

-rw-r--r--. 1 root root 3.9M Oct 17 01:48 traces/per_request/request_001.json.gz
-rw-r--r--. 1 root root 3.8M Oct 17 01:48 traces/per_request/request_002.json.gz
-rw-r--r--. 1 root root 3.8M Oct 17 01:48 traces/per_request/request_003.json.gz


In [ ]:
# Process: build merged module tree with kernel correlation
!python vllm_arch_map/profiler_to_diagram.py \
    --per-request-dir traces/per_request --out_dir samples/

Processing 3 per-request traces from traces/per_request...

[Req-001] request_001.json.gz
Loading traces/per_request/request_001.json.gz (4.0 MB)...
  187806 events | 2032 modules | 13623 cpu_ops | 2178 kernels

[Req-002] request_002.json.gz
Loading traces/per_request/request_002.json.gz (4.0 MB)...
  187676 events | 2032 modules | 13623 cpu_ops | 2178 kernels

[Req-003] request_003.json.gz
Loading traces/per_request/request_003.json.gz (4.0 MB)...
  187643 events | 2032 modules | 13614 cpu_ops | 2178 kernels

Wrote samples/per_request_profiles.json (2.4 MB)
  3 requests merged into one tree


---
## Tab 1: Workload Dashboard

Two terminals needed. Produces `workload_data.json`.

`nvtx_serve.py` wraps vLLM's server with NVTX markers around each request, so nsys can segment the GPU kernel timeline per request. The benchmark client generates load, and Prometheus provides server-side aggregate metrics. `parse_workload.py` merges all three sources into one JSON.

In [28]:
# ┌─────────────────────────────────────────────────────────────┐
# │ TERMINAL 1: Start server under nsys profiler               │
# │ Run this in a separate terminal, not in the notebook.      │
# └─────────────────────────────────────────────────────────────┘
#
import subprocess, os, time, urllib.request
os.makedirs("benchmark_outputs", exist_ok=True)

# Start server under nsys profiler in background
server = subprocess.Popen(
    ["nsys", "profile", "-t", "cuda,nvtx", "--cuda-memory-usage=true", "-o", "vllm_trace",
    "python", "vllm_arch_map/nvtx_serve.py", os.environ["MODEL"], "--max-model-len", "8192"],
    stdout=open("server_log.txt", "w"), stderr=subprocess.STDOUT
)

# Wait for server to be ready
for i in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health")
        print(f"Server ready under nsys! (pid={server.pid})")
        break
    except:
        time.sleep(2)
        if i % 10 == 9:
            print(f"Still waiting... ({(i+1)*2}s)")


# Wait for: "Uvicorn running on http://0.0.0.0:8000"

print("Run the nsys+server command in Terminal 1, then continue.")

Still waiting... (20s)
Still waiting... (40s)
Server ready under nsys! (pid=649924)
Run the nsys+server command in Terminal 1, then continue.


In [29]:
# ┌─────────────────────────────────────────────────────────────┐
# │ TERMINAL 2: Run benchmark + capture metrics                │
# └─────────────────────────────────────────────────────────────┘
import os
os.environ["MODEL"] = "Qwen/Qwen3-8B"

os.makedirs("benchmark_outputs", exist_ok=True)
!vllm bench serve --backend vllm --model $MODEL \
    --base-url http://localhost:8000 \
    --endpoint /v1/completions --dataset-name random \
    --num-prompts 3 --max-concurrency 1 \
    --random-input-len 1024 --random-output-len 128 \
    --save-result --save-detailed --result-dir benchmark_outputs/

2300.27s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7fd895ecb1a0>, trust_remote_code=False, seed=0, num_prompts=3, dataset_name='random', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=1024, random_output_len=128, random_range_ratio='0.0', random_prefix_len=0, random_batch_size=1, no_reranker=False, random_mm_base_items_per_request=1, random_mm_

In [30]:
# Capture Prometheus metrics (server still running)
!curl -s http://localhost:8000/metrics > benchmark_outputs/prometheus.txt
print("Prometheus metrics saved. Now Ctrl+C the server in Terminal 1.")
server.terminate()
server.wait()
print("Server stopped. nsys wrote vllm_trace.nsys-rep")

2326.67s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Prometheus metrics saved. Now Ctrl+C the server in Terminal 1.
Server stopped. nsys wrote vllm_trace.nsys-rep


In [23]:
# After Ctrl+C: export nsys trace to sqlite
!nsys export --type=sqlite vllm_trace.nsys-rep --force-overwrite true

2080.67s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Processing 3088234 events: [===============================================100%]


In [31]:
# Merge benchmark JSON + Prometheus + nsys sqlite
!ls benchmark_outputs
!python vllm_arch_map/parse_workload.py \
    --bench benchmark_outputs/*.json \
    --prometheus benchmark_outputs/prometheus.txt \
    --nsys vllm_trace.sqlite \
    -o samples/workload_data.json

2356.57s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


prometheus.txt	vllm-infqps-concurrency1-Qwen3-8B-20261017-022447.json


2361.75s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Auto-detected 36 layers from Qwen/Qwen3-8B
Wrote samples/workload_data.json (19.9 KB)


---
## Tab 3: Architecture (one-time per model)

Single terminal. Produces `model_execution.json`.

`walk_model.py` walks the `nn.Module` tree and captures a PyTorch profiler trace. The trace is saved as a compressed `.json.gz` file (raw Chrome trace format). `profiler_to_diagram.py` reads it directly (no manual decompression needed) and builds the module hierarchy with kernel mappings.

In [32]:
# Walk model tree + capture profiler trace
!VLLM_ALLOW_INSECURE_SERIALIZATION=1 python vllm_arch_map/walk_model.py $MODEL \
    -o samples/model_tree.json --with-forward --profile-dir ./samples/traces/

2381.30s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


(MainProcess pid=651405) INFO 10-17 02:26:04 [api_utils.py:286] non-default args: {'max_model_len': 128, 'gpu_memory_utilization': 0.5, 'disable_log_stats': True, 'enforce_eager': True, 'profiler_config': ProfilerConfig(profiler='torch', torch_profiler_dir='/root/lkesem/vllm_arch_map/samples/traces/profiles-2c8eb9ca-a671-41bb-b1d4-5e22bdbad421', torch_profiler_activities=None, proton_profiler_dir='', proton_context='shadow', proton_data='tree', proton_backend=None, proton_mode=None, proton_hook=None, proton_output_format=None, proton_graph_attribution=False, torch_profiler_with_stack=True, torch_profiler_with_flops=False, torch_profiler_use_gzip=True, torch_profiler_dump_cuda_time_total=True, torch_profiler_record_shapes=True, torch_profiler_with_memory=False, capture_torch_profiler=False, detailed_trace_annotation=False, ignore_frontend=False, delay_iterations=0, max_iterations=0, warmup_iterations=0, active_iterations=5, wait_iterations=0), 'model': 'Qwen/Qwen3-8B'}
(MainProcess pid=

In [33]:
# Process: build module tree with kernel correlation + layer collapse
!python vllm_arch_map/profiler_to_diagram.py \
    --out_dir samples/ --traces-dir samples/traces

2461.06s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Processing rank0.1789640479161371744.pt.trace.json...
Loading samples/traces/profiles/rank0.1789640479161371744.pt.trace.json (61.5 MB)...
  171176 events | 5080 modules | 40598 cpu_ops | 4575 kernels
Wrote samples/model_execution.json (2.7 MB)
  module_tree: Qwen3ForCausalLM | 36 layers | 30 kernel types


---
## View

### On the GPU host
```bash
python3 -m http.server 8765 --directory samples
```

### Or SCP to local machine
```bash
scp host:~/vllm_arch_map/samples/per_request_profiles.json samples/
scp host:~/vllm_arch_map/samples/workload_data.json samples/
scp host:~/vllm_arch_map/samples/model_execution.json samples/

python3 -m http.server 8765 --directory samples
open http://localhost:8765/model_viz.html
```

---
## Output Files

| File | Tab | Size | Contents |
|---|---|---|---|
| `per_request_profiles.json` | Tab 2 | ~1 MB | Merged module tree with per-request timing, kernel mapping, layer variance |
| `workload_data.json` | Tab 1 | ~50 KB | Per-request TTFT/decode/queue, GPU kernels, Prometheus metrics |
| `model_execution.json` | Tab 3 | ~300 KB | Module hierarchy, op→kernel correlation, layer collapse |

Tested on **DGX SPARK** (NVIDIA H100 80GB HBM3) with **Qwen/Qwen3-8B**.